# Quantify Superadditivity

Measures superadditive interaction effects between data heterogeneity and network topology in decentralized federated learning (D-SGD).

**Requirements**: A100 GPU runtime on Google Colab.

On the first run, datasets download from the internet and are cached on Google Drive so subsequent sessions skip the download entirely.

## 1. Environment Setup

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> A100 GPU.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone Repo & Install Dependencies

In [ ]:
import os
if not os.path.isdir('/content/quantify_superadditivity'):
    !git clone https://github.com/angelafoua/quantify_superadditivity /content/quantify_superadditivity
%cd /content/quantify_superadditivity
!git checkout main && git pull origin main

In [ ]:
!pip install -q hydra-core==1.3.2 omegaconf==2.3.0 h5py networkx statsmodels seaborn tabulate tqdm scikit-learn pandas scipy
!pip install -q -e . --no-deps
!python scripts/validate_setup.py

## 3. Cache Data & Outputs on Google Drive

Symlinks `data/` and `outputs/` to Google Drive so datasets download once and results persist across Colab sessions.

In [ ]:
import os

# --- Cache datasets on Google Drive ---
DRIVE_DATA = '/content/drive/MyDrive/superadditivity_data'
LOCAL_DATA = '/content/quantify_superadditivity/data'

os.makedirs(DRIVE_DATA, exist_ok=True)

if os.path.isdir(LOCAL_DATA) and not os.path.islink(LOCAL_DATA):
    !cp -rn {LOCAL_DATA}/* {DRIVE_DATA}/ 2>/dev/null || true
    !rm -rf {LOCAL_DATA}

if os.path.islink(LOCAL_DATA):
    os.remove(LOCAL_DATA)

os.symlink(DRIVE_DATA, LOCAL_DATA)
print('data ->', os.path.realpath(LOCAL_DATA))

# --- Cache outputs on Google Drive ---
DRIVE_OUT = '/content/drive/MyDrive/superadditivity_outputs'
os.makedirs(DRIVE_OUT, exist_ok=True)
if os.path.islink('outputs') or os.path.exists('outputs'):
    !rm -rf outputs
os.symlink(DRIVE_OUT, 'outputs')
print('outputs ->', os.path.realpath('outputs'))

## 4. Quick Validation (1 seed, 4 runs)

Confirms the pipeline runs end-to-end and that the interaction term I is directionally present before committing GPU-hours on the full sweep.

In [ ]:
!python scripts/run_sweep.py --experiment core_factorial_quick --resume

## 5. Full Factorial Sweep (5 run_seeds x 3 graph_seeds = 60 runs)

15 replicates per cell of the 2x2 factorial (IID/Non-IID x Dense/Community) for statistical power.

In [ ]:
!python scripts/run_sweep.py --experiment core_factorial --resume

## 6. Analysis & Visualisation

In [ ]:
!python scripts/run_analysis.py --results_dir outputs/core_factorial_quick
!python scripts/run_visualization.py --results_dir outputs/core_factorial_quick

In [ ]:
!python scripts/run_analysis.py --results_dir outputs/core_factorial
!python scripts/run_visualization.py --results_dir outputs/core_factorial